# SISTEMAS DE RECOMENDACIÓN 2
## PRÁCTICA

<img src="portada.jpg" align="center" width="80%"/>

El principal objetivo de cualquier sistema de recomendación es ofrecer a los usuarios una experiencia personalizada. ¿Cómo traducimos esto a un problema de machine learning? Tenemos dos tipos de problemas:

- Problema de predicción: el primer enfoque es aquel en el que nos gustaría predecir el valor de calificación de una combinación usuario-artículo con la suposición de que se dispone de datos de entrenamiento que indican la preferencia de un usuario por otros artículos. Imaginemos una matriz m × n, donde m es el número de usuarios y n el número de elementos, y el objetivo es predecir los valores que faltan (o no observados). En otras palabras, **queremos predecir el rating que un usuario preexistente le pondría a un ítem que aún no consumió.**
<br>

- Problema de clasificación: en otras ocasiones, deseamos **recomendar los k mejores artículos para un usuario concreto o determinar los k mejores usuarios a los que dirigirnos para un ítem específico**. Este problema también se denomina problema de recomendación top-k y es la formulación de clasificación del problema de recomendación. Pensemos en un motor de búsqueda en el que, en función de quién realice la búsqueda, nos gustaría mostrar los elementos top-k para ofrecer resultados personalizados basados en sus preferencias anteriores y en su actividad reciente.

Clasficación de sistemas de recomendación 

<img src="sintesis.jpg" align="center" width="60%"/>

---
Si tienen problemas para instalar surprise
1. Instalar Microsoft Visual C++: https://visualstudio.microsoft.com/es/downloads/?q=build+tools
2. Desde la consola:
   - pip cache purge
   - pip install --upgrade pip setuptools wheel cython
   - pip install --no-cache-dir scikit-surprise==1.1.4
---


In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split as skl_train_test_split
from surprise import KNNBasic,NormalPredictor,BaselineOnly,NMF

%matplotlib inline

## Ejemplo práctico de un SR basado en contenido

Utilizamos el dataset goodbooks-10k (disponible en Kaggle):
- El primer CSV contiene datos de 10k libros puntuados por 53k usuarios.
- El segundo CSV contiene los metadatos (título, autor, ISBN, etc) para cada uno de los 10k libros.

In [58]:
ratings_data = pd.read_csv('./data/books/ratings.csv.zip')
books_metadata = pd.read_csv('./data/books/books.csv.zip')
ratings_data.head(10)

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
5,1,2077,4
6,1,2487,4
7,1,2900,5
8,1,3662,4
9,1,3922,5


In [59]:
ratings_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 981756 entries, 0 to 981755
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   book_id  981756 non-null  int64
 1   user_id  981756 non-null  int64
 2   rating   981756 non-null  int64
dtypes: int64(3)
memory usage: 22.5 MB


In [60]:
books_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         10000 non-null  int64  
 1   book_id                    10000 non-null  int64  
 2   best_book_id               10000 non-null  int64  
 3   work_id                    10000 non-null  int64  
 4   books_count                10000 non-null  int64  
 5   isbn                       9300 non-null   object 
 6   isbn13                     9415 non-null   float64
 7   authors                    10000 non-null  object 
 8   original_publication_year  9979 non-null   float64
 9   original_title             9415 non-null   object 
 10  title                      10000 non-null  object 
 11  language_code              8916 non-null   object 
 12  average_rating             10000 non-null  float64
 13  ratings_count              10000 non-null  int6

Tenemos además un dataset de tags, que nos da detalles sobre el argumento del libro.

In [61]:
book_tags = pd.read_csv('./data/books/book_tags.csv')
tags = pd.read_csv('./data/books/tags.csv')

In [62]:
data = pd.merge(book_tags, tags, left_on='tag_id', right_on='tag_id', how='inner')
titles = books_metadata[['id', 'book_id','title','authors']]
data = pd.merge(titles, data, left_on='book_id', right_on='goodreads_book_id')
data.drop(['goodreads_book_id', 'tag_id', 'count'], axis=1,inplace=True)
data.head(3)

,id,book_id,title,authors,tag_name
0,1,2767052,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,favorites
1,1,2767052,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,currently-reading
2,1,2767052,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,young-adult


In [63]:
list_tags = data.groupby(by='book_id')['tag_name'].apply(set).apply(list)
books_metadata['tags'] = books_metadata['book_id'].apply(lambda x: ' '.join(list_tags[x]))
pd.reset_option('max_colwidth')
books_metadata.head(3)

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tags
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,own-it book-club ebooks ya-dystopian default c...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,own-it kids default young-adult re-read childr...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,own-it book-club stephanie-meyer already-read ...


In [64]:
books_metadata['tags'][0]

'own-it book-club ebooks ya-dystopian default completed-series read-2011 young-adult re-read the-hunger-games audiobooks favorite-books fantasy owned favourites survival books-i-own trilogy to-buy faves post-apocalyptic borrowed favourite favorite-series 2012-reads dystopias reread romance e-book my-library english ebook 5-star to-read scifi read-in-2012 audio dystopian ya-books future sf sci-fi-fantasy kindle coming-of-age my-books dystopian-fiction read-in-2010 my-favorites books speculative-fiction fantasy-sci-fi ya-fantasy dystopia favorites distopia drama shelfari-favorites ya-fiction young-adult-fiction audiobook ya read-in-2013 library currently-reading ya-lit fiction suspense all-time-favorites sci-fi read-2012 favorite action novels i-own read-in-2014 finished series scifi-fantasy read-more-than-once action-adventure novel favs favourite-books loved thriller reviewed love-triangle teen hunger-games futuristic suzanne-collins teen-fiction distopian love science-fiction contempo

In [65]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

Al trabajar con texto, podemos aplicar una transformación llamada **TF-IDF**, o Term Frequency-Inverse Document Frecuency. 

El **número de features** que crea es igual al número total de palabras distintas utilizadas en la columna de tags. 

Los **valores** son directamente *proporcionales al número de veces que se utiliza una palabra concreta* e *inversamente proporcionales al número de documentos (libros en este caso) en los que se utiliza la palabra*.

Penalizará una palabra presente en los tags de un libro si es común a muchos libros. Las palabras que aparecen varias veces pero son comunes a muchos libros no son tan útiles para diferenciar los distintos libros.

En otras palabras, **el proceso TF-IDF nos dice que tan relevante es una palabra a un documento, en una colección de documentos**.

### Cálculo

t - término (palabra) <br>
d - documento (conjunto de palabras) <br>
N - cantidad de documentos
<br>

tf(t,d) = ocurrencia de t en d / cantidad de palabras en d

df(t) = ocurrencia de t en N documentos

idf(t) = N/df

---
- IDF es la inversa de la frecuencia de documentos que mide que **tan informativo es el término t**. 
- Cuando calculamos IDF, será muy bajo para las palabras más frecuentes, como las stop words (porque están presentes en casi todos los documentos, y N/df dará un valor muy bajo a esa palabra). Esto da finalmente lo que queremos, un peso relativo.
---

Comparemos dos palabras con idéntico tf pero diferente idf:


N = 5
<br>
t = 3
<br>
d = 10
<br>
tf(t,d) = 3/10


palabra 1 (más común): 
<br>
df(t) = 5
<br>
idf(t) = 5/5 = 1

tf-idf* = 0.3*1 = 0.3

palabra 2 (más rara):
<br>
df(t) = 2
<br>
idf(t) = 5/2 = 2.5
<br>
tf-idf* = 0.3*2.5 = 0.75
<br>

Vemos que la segunda palabra tiene *un mayor valor tf-idf al ser más rara*.
<br>
Por otro lado, se suele usar el logaritmo para contrarrestar el efecto de un N muy grande: idf(t) = log(N/(df + 1)).

El score final será la multiplicación de ambos términos:

**tf-idf(t, d) = tf(t, d) * log(N/(df + 1))**

In [66]:
tfidf = TfidfVectorizer(stop_words='english')

tags_matrix = tfidf.fit_transform(books_metadata['tags'])

tags_matrix.shape

(10000, 16396)

In [67]:
tags_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 767020 stored elements and shape (10000, 16396)>

In [68]:
#Cantidad de veces que se repite cada término
tfidf.vocabulary_

{'book': 2049,
 'club': 3089,
 'ebooks': 4490,
 'ya': 15260,
 'dystopian': 4451,
 'default': 3899,
 'completed': 3269,
 'series': 12640,
 'read': 11635,
 '2011': 251,
 'young': 15308,
 'adult': 632,
 'hunger': 6863,
 'games': 5744,
 'audiobooks': 1336,
 'favorite': 5188,
 'books': 2075,
 'fantasy': 5132,
 'owned': 10424,
 'favourites': 5197,
 'survival': 13600,
 'trilogy': 14289,
 'buy': 2418,
 'faves': 5182,
 'post': 11099,
 'apocalyptic': 1068,
 'borrowed': 2112,
 'favourite': 5196,
 '2012': 253,
 'reads': 11662,
 'dystopias': 4453,
 'reread': 11889,
 'romance': 12101,
 'library': 8325,
 'english': 4736,
 'ebook': 4487,
 'star': 13315,
 'scifi': 12480,
 'audio': 1333,
 'future': 5686,
 'sf': 12671,
 'sci': 12468,
 'fi': 5268,
 'kindle': 7818,
 'coming': 3228,
 'age': 686,
 'fiction': 5289,
 '2010': 249,
 'favorites': 5190,
 'speculative': 13198,
 'dystopia': 4450,
 'distopia': 4179,
 'drama': 4328,
 'shelfari': 12754,
 'audiobook': 1334,
 '2013': 254,
 'currently': 3705,
 'reading': 

In [69]:
doc = 0
feature_index = tags_matrix[doc,:].nonzero()[1]
tfidf_scores = zip(feature_index, [tags_matrix[doc, x] for x in feature_index])

In [70]:
books_metadata['tags'][0]

'own-it book-club ebooks ya-dystopian default completed-series read-2011 young-adult re-read the-hunger-games audiobooks favorite-books fantasy owned favourites survival books-i-own trilogy to-buy faves post-apocalyptic borrowed favourite favorite-series 2012-reads dystopias reread romance e-book my-library english ebook 5-star to-read scifi read-in-2012 audio dystopian ya-books future sf sci-fi-fantasy kindle coming-of-age my-books dystopian-fiction read-in-2010 my-favorites books speculative-fiction fantasy-sci-fi ya-fantasy dystopia favorites distopia drama shelfari-favorites ya-fiction young-adult-fiction audiobook ya read-in-2013 library currently-reading ya-lit fiction suspense all-time-favorites sci-fi read-2012 favorite action novels i-own read-in-2014 finished series scifi-fantasy read-more-than-once action-adventure novel favs favourite-books loved thriller reviewed love-triangle teen hunger-games futuristic suzanne-collins teen-fiction distopian love science-fiction contempo

In [71]:
feature_names = tfidf.get_feature_names_out()
tfidf_scores_sorted=[]
for w, s in [(feature_names[i], s) for (i, s) in tfidf_scores][:10]:
    tfidf_scores_sorted.append([w,s])
    print(w,s)

stars 0.044902535151457965
contemporary 0.03622900456869731
science 0.04945565646920866
distopian 0.12208063904468137
collins 0.1627826657237944
suzanne 0.16895997553164158
futuristic 0.1022233545057037
teen 0.121055818234319
triangle 0.09451559019735938
love 0.11821827337892689


In [72]:
tfidf_scores_sorted_df=pd.DataFrame(tfidf_scores_sorted)
tfidf_scores_sorted_df.columns=['token','val']
tfidf_scores_sorted_df.sort_values(by='val',ascending=False,inplace=True)
tfidf_scores_sorted_df.head(10)

,token,val
5,suzanne,0.168960
4,collins,0.162783
3,distopian,0.122081
7,teen,0.121056
9,love,0.118218
6,futuristic,0.102223
8,triangle,0.094516
2,science,0.049456
0,stars,0.044903
1,contemporary,0.036229


¿Que palabras comunes se les ocurre que podríamos sacar?

### Similitud Coseno

<img src="similitud_coseno.jpg" align="center" width="60%"/>

Así como vimos la distancia euclideana y el coeficiente de Pearson para estimar la similitud entre dos personas o ítems, existe otra medida muy utilizada llamada *similitud coseno*. En este caso la utilizaremos para estimar la similitud entre los contenidos de los ítems.

La imagen muestra la similitud coseno en dos dimensiones. Si dos vectores de datos son cercanos, el ángulo entre estos dos vectores es pequeño y la similitud coseno será alta. En el extremo, dos vectores iguales tendrán un ángulo de 0 grados, un coseno de 1, y una similitud coseno también de 1.

Naturalmente, podemos llevar el concepto para N dimensiones.

In [73]:
similarity_matrix = cosine_similarity(tags_matrix,tags_matrix)
similarity_matrix

array([[1.        , 0.38961343, 0.40894373, ..., 0.09091467, 0.16148787,
        0.03602845],
       [0.38961343, 1.        , 0.37207812, ..., 0.0886315 , 0.16692497,
        0.04592596],
       [0.40894373, 0.37207812, 1.        , ..., 0.05919159, 0.09024158,
        0.02882952],
       ...,
       [0.09091467, 0.0886315 , 0.05919159, ..., 1.        , 0.05409486,
        0.12427748],
       [0.16148787, 0.16692497, 0.09024158, ..., 0.05409486, 1.        ,
        0.11579571],
       [0.03602845, 0.04592596, 0.02882952, ..., 0.12427748, 0.11579571,
        1.        ]])

In [74]:
def recommend_books_based_on_plot(book_input,n=15):
    book_index = books_metadata.loc[books_metadata.original_title==book_input,:].index[0]

    #get similarity values with other books
    #similarity_score is the list of index and similarity matrix
    similarity_score = list(enumerate(similarity_matrix[book_index]))

    #sort in descending order the similarity score of book inputted with all the other books
    similarity_score = sorted(similarity_score, key=lambda x: x[1], reverse=True)

    # Get the scores of the 15 most similar books. Ignore the first book.
    similarity_score = similarity_score[1:n]

    #return book names using the mapping series
    book_indices = [i[0] for i in similarity_score]

    return (books_metadata[['original_title','authors','tags']].iloc[book_indices])

In [75]:
recommend_books_based_on_plot('Nineteen Eighty-Four',10)

,original_title,authors,tags
54,Brave New World,Aldous Huxley,own-it book-club english-literature ebooks def...
13,Animal Farm: A Fairy Story,George Orwell,book-club english-literature required-reading ...
47,Fahrenheit 451,Ray Bradbury,own-it book-club required-reading ebooks read-...
27,Lord of the Flies,William Golding,own-it book-club english-literature required-r...
70,"Frankenstein; or, The Modern Prometheus","Mary Wollstonecraft Shelley, Percy Bysshe Shel...",own-it book-club british-lit english-literatur...
172,A Clockwork Orange,Anthony Burgess,own-it book-club british-lit ebooks banned you...
808,Brave New World/Brave New World Revisited,"Aldous Huxley, Christopher Hitchens",own-it book-club english-literature default ut...
7,The Catcher in the Rye,J.D. Salinger,own-it book-club required-reading read-again d...
845,Animal Farm & 1984,"George Orwell, Christopher Hitchens",own-it book-club english-literature ebooks rea...


In [76]:
recommend_books_based_on_plot('Nineteen Eighty-Four',15)

,original_title,authors,tags
54,Brave New World,Aldous Huxley,own-it book-club english-literature ebooks def...
13,Animal Farm: A Fairy Story,George Orwell,book-club english-literature required-reading ...
47,Fahrenheit 451,Ray Bradbury,own-it book-club required-reading ebooks read-...
27,Lord of the Flies,William Golding,own-it book-club english-literature required-r...
70,"Frankenstein; or, The Modern Prometheus","Mary Wollstonecraft Shelley, Percy Bysshe Shel...",own-it book-club british-lit english-literatur...
172,A Clockwork Orange,Anthony Burgess,own-it book-club british-lit ebooks banned you...
808,Brave New World/Brave New World Revisited,"Aldous Huxley, Christopher Hitchens",own-it book-club english-literature default ut...
7,The Catcher in the Rye,J.D. Salinger,own-it book-club required-reading read-again d...
845,Animal Farm & 1984,"George Orwell, Christopher Hitchens",own-it book-club english-literature ebooks rea...
64,"Slaughterhouse-Five, or The Children's Crusade...",Kurt Vonnegut Jr.,own-it book-club ebooks default re-read world-...


In [77]:
recommend_books_based_on_plot('Misery')

,original_title,authors,tags
910,Gerald's Game,Stephen King,own-it ebooks default stephen-king-collection ...
1074,Dolores Claiborne,"Stephen King, Dominique Dill",own-it book-club ebooks murder default fiction...
674,Firestarter,Stephen King,own-it ebooks default fiction-horror re-read s...
304,Pet Sematary,Stephen King,own-it ebooks default fiction-horror re-read m...
669,The Dead Zone,Stephen King,own-it ebooks default fiction-horror re-read s...
6322,NaN,Stephen King,ebooks fiction-horror stephen-king-to-read hor...
552,Needful Things,Stephen King,own-it ebooks default stephen-king-collection ...
1422,Rose Madder,Stephen King,own-it ebooks default stephen-king-collection ...
914,Insomnia,"Stephen King, Bettina Blanch Tyroller",own-it ebooks default stephen-king-collection ...
704,Thinner,"Richard Bachman, Stephen King",own-it ebooks default fiction-horror re-read s...


In [78]:
recommend_books_based_on_plot('The Hobbit')

,original_title,authors,tags
3520,Neverwhere Graphic Novel,"Mike Carey, Glenn Fabry, Neil Gaiman",own-it to-get ebooks graphic default young-adu...
963,The Hobbit and The Lord of the Rings,J.R.R. Tolkien,own-it five-stars already-read jrr-tolkien def...
154,The Two Towers,J.R.R. Tolkien,own-it sci-fi-and-fantasy jrr-tolkien ebooks d...
18,The Fellowship of the Ring,J.R.R. Tolkien,own-it sci-fi-and-fantasy ebooks default lord-...
7421,Coraline,"Neil Gaiman, P. Craig Russell",coraline ebooks graphic kids default young-adu...
6,The Hobbit or There and Back Again,J.R.R. Tolkien,own-it book-club sci-fi-and-fantasy ebooks jrr...
160,The Return of the King,J.R.R. Tolkien,own-it sci-fi-and-fantasy jrr-tolkien ebooks d...
8373,Darth Vader and Son,Jeffrey Brown,own-it space ebooks graphic kids star-wars def...
6737,The Hedge Knight,"Ben Avery, Mike S. Miller, George R.R. Martin",ebooks sci-fi-and-fantasy graphic default youn...
1697,The Dark Tower: The Gunslinger Born,"Peter David, Robin Furth, Jae Lee, Richard Isa...",to-read-comics own-it sci-fi-and-fantasy graph...


Para obtener recomendaciones más diversas, podríamos dejar afuera las novelas del mismo autor de la novela target.

In [79]:
recs_df=recommend_books_based_on_plot('The Hobbit')
recs_df.loc[~recs_df.authors.str.contains('J.R.R. Tolkien'),:]

,original_title,authors,tags
3520,Neverwhere Graphic Novel,"Mike Carey, Glenn Fabry, Neil Gaiman",own-it to-get ebooks graphic default young-adu...
7421,Coraline,"Neil Gaiman, P. Craig Russell",coraline ebooks graphic kids default young-adu...
8373,Darth Vader and Son,Jeffrey Brown,own-it space ebooks graphic kids star-wars def...
6737,The Hedge Knight,"Ben Avery, Mike S. Miller, George R.R. Martin",ebooks sci-fi-and-fantasy graphic default youn...
1697,The Dark Tower: The Gunslinger Born,"Peter David, Robin Furth, Jae Lee, Richard Isa...",to-read-comics own-it sci-fi-and-fantasy graph...
8382,A Wrinkle in Time: The Graphic Novel,"Hope Larson, Madeleine L'Engle",book-club space graphic kids young-adult child...
3634,Bone: The Complete Cartoon Epic in One Volume,Jeff Smith,to-read-comics own-it kids default young-adult...


## Ejemplo práctico de un SR basado en filtros colaborativos: Librería Surprise

Tenemos que crear un objeto dataset para trabajar con Surprise. Este dataset contiene los siguientes elementos:
1. Los IDs de los usuarios
2. Los IDs de cada ítem (en este caso, de cada libro)
3. El rating correspondiente (en este caso, en una escala del 1 al 5)


In [80]:
from surprise import Dataset
from surprise import Reader

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_data[['user_id', 'book_id', 'rating']], reader)

### Entrenamos un modelo SVD con crossvalidation

Podemos entrenar con cross-validation un modelo a partir del algoritmo SVD (descomposición de valor singular o Singular Value Decomposition) para construir un sistema de recomendación. SVD es un algoritmo de factorización de matrices que puede utilizarse para sistemas de recomendación.

Los sistemas de recomendación que utilizan la factorización matricial suelen seguir un patrón en el que una matriz de puntuaciones o ratings se factoriza en un producto de matrices que representan factores latentes para los ítems (en este caso, libros) y los usuarios. Funciona con la técnica de factorización matricial que vimos la clase pasada.

In [81]:
from surprise import SVD
from surprise.model_selection import cross_validate

svd = SVD(verbose=True, n_epochs=30,random_state=100)
cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20


KeyboardInterrupt: 

También podemos entrenar el modelo en todo el conjunto de datos utilizando el método fit después de convertir el objeto Dataset en un objeto Surprise Trainset utilizando el método build_full_trainset.

In [ ]:
trainset = data.build_full_trainset()
svd.fit(trainset)

Ahora podemos realizar predicciones para un ID de un libro específico, con el método predict.

In [ ]:
svd.predict(uid=10, iid=100)

Tomemos de ejemplo un fanático confeso de los libros de Dan Brown (autor de El Código Da Vinci). Observemos sus puntuaciones a los libros de dicho autor:

In [ ]:
books_metadata.loc[books_metadata.authors.str.contains('Dan Brown'),:]

In [ ]:
libros_db=books_metadata.loc[books_metadata.authors.str.contains('Dan Brown'),'id'].values
libros_db

In [ ]:
#Buscamos lectores amantes de Dan Brown
ratings_data_f=ratings_data.loc[(ratings_data.book_id.isin(libros_db)),:].copy()
ratings_data_f.groupby('user_id')[['book_id','rating']].agg({'book_id':'count','rating':'mean'}).sort_values('book_id',ascending=False)

In [ ]:
libros_leidos_usuario_db=ratings_data.loc[(ratings_data.user_id==32918)&(ratings_data.book_id.isin(libros_db)),:]
libros_leidos_usuario_db

Vemos que en general hace una muy buena evaluación de los libros de DB

In [ ]:
libros_db[~np.isin(libros_db,libros_leidos_usuario_db)]

In [ ]:
pendientes_db=libros_db[~np.isin(libros_db,libros_leidos_usuario_db)]
pred=[]
for libro in pendientes_db:
    pred_puntual=svd.predict(uid=32918, iid=libro).est
    print(pred_puntual)    
    pred.append(pred_puntual)

media_db=ratings_data.loc[ratings_data.book_id.isin(libros_db),"rating"].mean()    
print(f'Media ratings libros DB: {round(media_db,2)}')    
print(f'Media ratings puntuación estimada libros no leídos de DB por usuario 32918: {round(np.array(pred).mean(),2)}')    


Vemos que la puntuación estimada por nuestro modelo para el libro aún no leído de Dan Brown es muy superior al promedio.
<br>

### Generando recomendaciones

In [ ]:
import difflib
import random

def get_book_id(book_title, metadata):
    
    existing_titles = list(metadata['title'].values)
    closest_titles = difflib.get_close_matches(book_title, existing_titles)
    book_id = metadata[metadata['title'] == closest_titles[0]]['id'].values[0]
    return book_id

def get_book_info(book_id, metadata):
    
    book_info = metadata[metadata['id'] == book_id][['id', 'isbn', 
                                                    'authors', 'title', 'original_title']]
    return book_info.to_dict(orient='records')

def predict_review(user_id, book_title, model, metadata):
    
    book_id = get_book_id(book_title, metadata)
    review_prediction = model.predict(uid=user_id, iid=book_id)
    return review_prediction.est

def generate_recommendation(user_id, model, metadata, thresh=4):
    
    book_titles = list(metadata['title'].values)
    random.shuffle(book_titles)
    
    for book_title in book_titles:
        rating = predict_review(user_id, book_title, model, metadata)
        if rating >= thresh:
            book_id = get_book_id(book_title, metadata)
            return get_book_info(book_id, metadata)


La función generate_recommendation genera una recomendación de libros para un usuario iterando a través de la lista de títulos de libros y prediciendo las valoraciones de los usuarios para cada título hasta que encuentra un libro con una valoración igual o superior al umbral especificado que lo califica para ser recomendado a un usuario. 

In [ ]:
%%time
generate_recommendation(32918, svd, books_metadata, 4.9)

### Visualizando las similitudes entre los libros utilizando UMAP

In [ ]:
svd.qi.shape

Ahora podemos apreciar la similitud entre los libros a partir de la matriz de factorización resultante luego de aplicar el algoritmo SVD.

Esta matriz de 10.000 x 100 tiene un vector de 100 dimensiones para cada libro, que son demasiadas dimensiones para que podamos visualizarlas intuitivamente, pero podemos utilizar una técnica de reducción de la dimensionalidad para representar cada libro como un punto bidimensional en el espacio. 

Podemos utilizar UMAP (Uniform Manifold Approximation and Projection, una alternativa a PCA) para representar cada libro como un punto bidimensional.

In [ ]:
#!pip install umap-learn==0.5.7

In [ ]:
from umap import UMAP
import pandas as pd
import plotly.express as px

# ==========================================
# 1. Generar embeddings 2D con UMAP
# ==========================================

umap_model = UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=1
)

books_embedding = umap_model.fit_transform(svd.qi)

# ==========================================
# 2. Crear dataframe de proyección
# ==========================================

projection = pd.DataFrame(
    books_embedding,
    columns=['x', 'y']
)

projection['title'] = books_metadata['original_title']

# ==========================================
# 3. Scatter global
# ==========================================

fig = px.scatter(
    projection,
    x='x',
    y='y',
    hover_data=['title'],
    title='Books Embedding Projection (UMAP)'
)

fig.show()

# Opcional: exportar html
# fig.write_html("books_umap.html")

Podemos ver que los puntos que representan los 10.000 libros parecen seguir una distribución normal bidimensional. Podemos explicar esta distribución de la siguiente forma:

- Algunos libros pueden ser populares en general entre un amplio abanico de audiencias y, por tanto, corresponder a puntos en el centro de este diagrama de dispersión.
- Otros libros pueden pertenecer a géneros muy específicos, como las novelas de vampiros, de misterio o románticas, que son populares entre un público concreto. Estos libros pueden corresponder a puntos alejados del centro del diagrama.

In [ ]:
import plotly.express as px

def plot_books(titles, plot_name):

    book_indices = []

    for book in titles:

        idx = get_book_id(book, books_metadata)

        if idx is not None:
            book_indices.append(idx - 1)

    book_vector_df = projection.iloc[book_indices]

    fig = px.scatter(
        book_vector_df,
        x='x',
        y='y',
        text='title',
        hover_data=['title'],
        title=plot_name
    )

    fig.update_traces(
        textposition='top center'
    )

    fig.show()

    # Opcional:
    # fig.write_html(f"{plot_name}.html")

¿Qué pasa si observamos algunos libros específicos? ¿Su similitud de características se ve reflejada en el nuevo espacio de dimensiones?

In [ ]:
books = list(books_metadata['title'][:40])
plot_books(books, plot_name='books_embedding')

In [94]:
from collections import defaultdict
from surprise import Dataset, SVD

def get_top_n(predictions, n=10):
    """Return the top-N recommendation for each user from a set of predictions.

    Args:
        predictions(list of Prediction objects): The list of predictions, as
            returned by the test method of an algorithm.
        n(int): The number of recommendation to output for each user. Default
            is 10.

    Returns:
    A dict where keys are user (raw) ids and values are lists of tuples:
        [(raw item id, rating estimation), ...] of size n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

In [ ]:
# algo = SVD()
# algo.fit(trainset)

In [ ]:
# Predice los ratings para todo par (u, i) que no aparece en train
# testset = trainset.build_anti_testset()
# predictions = algo_baseline.test(testset)

# Recomendamos usar un subsample porque las combinaciones de libros/usuarios que no aparecen en el set de
# train es muy grande!


In [ ]:
#Predecimos los puntajes de los libros no leídos por el usuario amante de DB
libros_leidos_usuario=ratings_data.loc[(ratings_data.user_id==32918),'book_id']
libros_no_leidos_usuario=ratings_data.loc[~ratings_data.book_id.isin(libros_leidos_usuario),'book_id'].unique()

pred=[]
for libro in libros_no_leidos_usuario:
    pred.append(svd.predict(uid=32918, iid=libro))

pred

In [ ]:
top_n = get_top_n(pred, n=10)
top_n

In [ ]:
# Predecimos el TOP-10 de recomendaciones para el usuario 32918

for uid, user_ratings in top_n.items():
    print([books_metadata.loc[books_metadata.id==iid,'original_title'].values[0] for (iid, _) in user_ratings])

#En el TOP 10 de libros recomendados para nuestro usuario de prueba, vemos libros de ciencia ficción y fantasía
#lo cual guarda sentido con su gusto con los libros de Dan Brown

### Comparamos el desempeño de varios algoritmos

In [ ]:
ratings_data_f=ratings_data.sample(n=100000)

In [ ]:
#CUIDADO! Mejor utilizar el método train_test_split de SK Learn, y luego convertir el conjunto de datos resultante al objeto Dataset
train, test = skl_train_test_split(ratings_data_f, test_size=0.20,random_state=100)
reader = Reader(rating_scale=(1, 5))
data_train = Dataset.load_from_df(train, reader)

algo_random=NormalPredictor()
algo_baseline=BaselineOnly()
algo_svd = SVD(random_state=100)
algo_knn = KNNBasic(random_state=100)
algo_nmf=NMF(random_state=100)

Nota: el algoritmo baseline tiene en cuenta únicamente los ratings promedio de usuarios e ítems, sin tener en cuenta las interacciones entre ellos.

In [ ]:
cv_random=cross_validate(algo_random, data_train, measures=['RMSE'],cv=4, verbose=False, n_jobs=-1)
cv_baseline=cross_validate(algo_baseline, data_train, measures=['RMSE'],cv=4, verbose=False, n_jobs=-1)
cv_svd=cross_validate(algo_svd, data_train, measures=['RMSE'],cv=4, verbose=False, n_jobs=-1)
cv_knn=cross_validate(algo_knn, data_train, measures=['RMSE'],cv=4, verbose=False, n_jobs=-1)
cv_nmf=cross_validate(algo_nmf, data_train, measures=['RMSE'],cv=4, verbose=False, n_jobs=-1)

In [ ]:
res=[]
res.append(round(cv_random['test_rmse'].mean(),3))
res.append(round(cv_baseline['test_rmse'].mean(),3))
res.append(round(cv_svd['test_rmse'].mean(),3))
res.append(round(cv_knn['test_rmse'].mean(),3))
res.append(round(cv_nmf['test_rmse'].mean(),3))

res_df=pd.DataFrame(pd.concat([pd.Series(['Random','Baseline','SVD','KNN','NMF']),pd.Series(res)],axis=1))
res_df.columns=['Algoritmo','RMSE CV']
res_df.sort_values(by='RMSE CV')

Los mejores algoritmos parecen ser el baseline y el SVD.

Ver tiempos de ajuste y rendimiento esperado en https://surpriselib.com/

### Métricas específicas de los sistemas de recomendación

<img src="metricas.png" align="center" width="60%"/>

- Un elemento se considera **relevante** si su puntuación es superior a un umbral determinado (por ej, mayor o igual a 4). En otras palabras, binarizamos el puntaje para convertir un problema de regresión en uno de clasificación.
<br>

- Un elemento se considera **recomendado** si se encuentra entre las k mejores puntuaciones estimadas.
<br>

- Tanto en precision@k como en recall@k el numerador vendrá dado por los ítems recomendados relevantes, vale decir, **los aciertos del modelo**.

Tener en cuenta que en los casos extremos en los que se produce una división por cero, los valores Precision@k y Recall@k no están definidos. Por convención, fijamos sus valores en 0 en estos casos.

In [ ]:
from collections import defaultdict

from surprise import Dataset, SVD
from surprise.model_selection import KFold


def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Return precision and recall at k metrics for each user"""

    # First map the predictions to each user.
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():

        # Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )

        # Precision@K: Proportion of recommended items that are relevant
        # When n_rec_k is 0, Precision is undefined. We here set it to 0.

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Recall@K: Proportion of relevant items that are recommended
        # When n_rel is 0, Recall is undefined. We here set it to 0.

        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls

In [ ]:
kf = KFold(n_splits=4)
algo = SVD(n_epochs=10, random_state=100)
precision_3=[]
precision_10=[]
recall_3=[]
recall_10=[]

for trainset, testset in kf.split(data):
    algo.fit(trainset)
    predictions = algo.test(testset)

    #Calculamos en promedio los valores de recall@k y precision@k para cada usuario
    
    precisions, recalls = precision_recall_at_k(predictions, k=3, threshold=4)
    precision_3.append(sum(prec for prec in precisions.values()) / len(precisions))
    recall_3.append(sum(rec for rec in recalls.values()) / len(recalls))
    
    precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=4)
    precision_10.append(sum(prec for prec in precisions.values()) / len(precisions))
    recall_10.append(sum(rec for rec in recalls.values()) / len(recalls))


In [ ]:
#Top 10
print(f'Precision @10 medio: {round(np.array(precision_10).mean(),2)}')
print(f'Recall @10 medio: {round(np.array(recall_10).mean(),2)}')

Esto significa que, en promedio, el 48% del top 10 de recomendaciones hechas para cada usuario son relevantes (con un puntaje estimado mayor o igual a 4), mientras que de todos los libros relevantes, el 35% aparecieron en el top 10.

--------

Podríamos reducir k, lo que resultaría en **mayor precision a costa de una menor recall**.
<br>

Esto es equivalente a utilizar un punto de corte mayor (= más exigentes) en un modelo convencional de ML.

In [ ]:
#Top 3
print(f'Precision @3 medio: {round(np.array(precision_3).mean(),2)}')
print(f'Recall @3 medio: {round(np.array(recall_3).mean(),2)}')


# PRÁCTICA

A partir del dataset de películas disponible en https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset/data, construir dos sistemas de recomendación:
- uno basado en contenidos a partir de la columna "overview" (argumento de la película). Generar recomendaciones de películas similares a Toy Story.
- un modelo basado en filtros colaborativos a partir de los ratings. Generar 10 recomendaciones para un fanático de Star Wars.

Además de lo visto en clase, puede ayudarse a partir de lo que aparece acá:
https://www.datacamp.com/tutorial/recommender-systems-python

TAREA para el hogar: construir una función para medir precision@k y recall@k a partir del listado de recomendaciones del SR basado en contenidos.

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
ratings_metadata = pd.read_csv('./data/movies/ratings_small.csv', low_memory=False)
movies_metadata = pd.read_csv('./data/movies/movies_metadata.csv', low_memory=False)
movies_metadata.id=pd.to_numeric(movies_metadata.id,errors='coerce')
movies_metadata.imdb_id=pd.to_numeric(movies_metadata.imdb_id,errors='coerce')

print(ratings_metadata.shape)
ratings_metadata.head()

In [ ]:
links = pd.read_csv('./data/movies/links.csv', low_memory=False)
links

In [ ]:
links.dtypes

In [ ]:
movies_metadata=movies_metadata.merge(links,left_on='id',right_on='tmdbId')

In [ ]:
movies_metadata.rename(columns={'movieId':'ratingsId'},inplace=True)
movies_metadata.drop(columns=['id','imdb_id','tmdbId'],inplace=True)
movies_metadata.dropna(subset='overview',inplace=True)

In [ ]:
movies_metadata.head()

In [ ]:
movies_metadata.loc[movies_metadata.vote_count>=1000,'original_title']

In [ ]:
ratings_metadata=ratings_metadata.loc[ratings_metadata.movieId.isin(movies_metadata.ratingsId),:]
ratings_metadata.shape

## Basado en contenidos

In [ ]:
%%time
# movies_metadata_f=movies_metadata.sample(10000)
tfidf = TfidfVectorizer(stop_words='english')
overview_matrix = tfidf.fit_transform(movies_metadata['overview'])
similarity_matrix = cosine_similarity(overview_matrix,overview_matrix)
print(similarity_matrix.shape)

In [ ]:
movie_index = movies_metadata.loc[movies_metadata.title=='Toy Story',:].index[0]
movie_index

### v1

In [ ]:
def recommend_movies_based_on_plot(movie_input,n=15):
    movie_index = movies_metadata.loc[movies_metadata.title==movie_input,:].index[0]

    #get similarity values with other movies
    #similarity_score is the list of index and similarity matrix
    similarity_score = list(enumerate(similarity_matrix[movie_index]))

    #sort in descending order the similarity score of movie inputted with all the other movies
    similarity_score = sorted(similarity_score, key=lambda x: x[1], reverse=True)

    # Get the scores of the 15 most similar movies. Ignore the first movie.
    similarity_score = similarity_score[1:n]

    #return movie names using the mapping series
    movie_indices = [i[0] for i in similarity_score]

    return (movies_metadata[['title','genres','overview']].iloc[movie_indices])

In [ ]:
recommend_movies_based_on_plot('Toy Story',10)

Potenciales mejoras:
1. Como es esperable, se recomiendan las secuelas de la película. Los  nombres Andy y Woody parece ser palabras que definen las películas recomendadas, por lo que convendría sacarlas al momento de hacer el ajuste del modelo TFIDF.
2. Además, podemos filtrar solo la recomendación de películas más o menos conocidas.
3. Por último, podemos utilizar ponderar al género como parte del contenido, a fin de que recomiende películas algo más similares.

### v2

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import numpy as np
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import TfidfVectorizer

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(["andy","woody","buzz"]))

# ── Precomputar las matrices base (una sola vez) ──────────────────
tfidf = TfidfVectorizer(ngram_range=(1,1), stop_words=my_stop_words)
plot_matrix  = normalize(tfidf.fit_transform(movies_metadata['overview']))  # sparse, normalizada

mlb = MultiLabelBinarizer()
genre_matrix = normalize(mlb.fit_transform(movies_metadata['genres']).astype(float))  # normalizada

WEIGHT_PLOT  = 0.5
WEIGHT_GENRE = 0.5

# ── Función de recomendación ──────────────────────────────────────
def recommend_movies_based_on_plot_v2(movie_input, n=15):
    movie_index = movies_metadata.loc[movies_metadata.title == movie_input, :].index[0]

    # Similitud solo para la fila de la película consultada (vector × matriz)
    plot_scores  = (plot_matrix[movie_index]  @ plot_matrix.T).toarray().flatten()
    genre_scores = (genre_matrix[movie_index] @ genre_matrix.T).flatten()

    combined_scores = WEIGHT_PLOT * plot_scores + WEIGHT_GENRE * genre_scores

    # Ordenar excluyendo la película misma
    similarity_score = sorted(enumerate(combined_scores), key=lambda x: x[1], reverse=True)
    similarity_score = similarity_score[1:]

    filtered_scores = []
    for idx, score in similarity_score:
        row = movies_metadata.iloc[idx]
        overview = str(row['overview']).lower()

        #Excluimos películas con palabras de la lista
        has_excluded_word = any(word in overview for word in EXCLUDE_WORDS)
        #Filtramos películas conocidas / populares
        is_popular = row['vote_count'] >= 1000

        if not has_excluded_word and is_popular:
            filtered_scores.append((idx, score))
        if len(filtered_scores) == n - 1:
            break

    movie_indices = [i[0] for i in filtered_scores]
    return movies_metadata[['title', 'genres', 'overview', 'vote_count']].iloc[movie_indices]

In [ ]:
EXCLUDE_WORDS = ["andy", "woody", "buzz"]
recomendaciones=recommend_movies_based_on_plot_v2('Toy Story',10)
recomendaciones

Ahora se obtienen películas más apuntadas al público infantil, en su mayoría de animación.

## Filtros colaborativos

In [82]:
train, test = skl_train_test_split(ratings_metadata, test_size=0.20,random_state=100)

In [83]:
from surprise import Dataset
from surprise import Reader

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_metadata[['userId', 'movieId', 'rating']], reader)
# data = Dataset.load_from_df(ratings_f[['userId', 'movieId', 'rating']], reader)

In [84]:
from surprise import SVD
from surprise.model_selection import cross_validate

svd = SVD(verbose=True, n_epochs=50,random_state=100)
cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 30
Processing epoch 31
Processing epoch 32
Processing epoch 33
Processing epoch 34
Processing epoch 35
Processing epoch 36
Processing epoch 37
Processing epoch 38
Processing epoch 39
Processing epoch 40
Processing epoch 41
Processing epoch 42
Processing epoch 43
Processing epoch 44
Processing epoch 45
Processing epoch 46
Processing epoch 47
Processing epoch 48
Processing epoch 49
Processing

{'test_rmse': array([0.91609392, 0.91604591, 0.91370908]),
 'test_mae': array([0.70609124, 0.70421449, 0.70387164]),
 'fit_time': (3.4393794536590576, 3.718966484069824, 3.5382845401763916),
 'test_time': (0.3400309085845947, 0.3702871799468994, 0.9389505386352539)}

In [85]:
#Entrenamos con todo el set de train
trainset = data.build_full_trainset()
svd.fit(trainset)  # ← esto es lo que falta

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 30
Processing epoch 31
Processing epoch 32
Processing epoch 33
Processing epoch 34
Processing epoch 35
Processing epoch 36
Processing epoch 37
Processing epoch 38
Processing epoch 39
Processing epoch 40
Processing epoch 41
Processing epoch 42
Processing epoch 43
Processing epoch 44
Processing epoch 45
Processing epoch 46
Processing epoch 47
Processing epoch 48
Processing epoch 49


In [87]:
#Obtenemos películas de Star Wars
movies_metadata.loc[movies_metadata.original_title.str.contains('Star Wars|The Empire Strikes Back|Return of the Jedi'),:]

,adult,belongs_to_collection,budget,genres,homepage,original_language,original_title,overview,popularity,poster_path,...,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,ratingsId,imdbId
256,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",11000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,en,Star Wars,Princess Leia is captured and held hostage by ...,42.149697,/btTdmkgIvOi0FFip1sPuZI2oQG6.jpg,...,121.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,"A long time ago in a galaxy far, far away...",Star Wars,False,8.1,6778.0,260,76759
1157,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",18000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,en,The Empire Strikes Back,"The epic saga continues as Luke Skywalker, in ...",19.470959,/6u1fYtxG5eqjhtCPDx04pJphQRW.jpg,...,124.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The Adventure Continues...,The Empire Strikes Back,False,8.2,5998.0,1196,80684
1170,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",32350000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,en,Return of the Jedi,As Rebel leaders map their strategy for an all...,14.586087,/jx5p0aHlbPXqe3AH9G15NvmWaqQ.jpg,...,135.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The Empire Falls...,Return of the Jedi,False,7.9,4763.0,1210,86190
2518,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",115000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,en,Star Wars: Episode I - The Phantom Menace,"Anakin Skywalker, a young slave strong with th...",15.649091,/n8V09dDc02KsSN6Q4hC2BX6hN8X.jpg,...,136.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Every generation has a legend. Every journey h...,Star Wars: Episode I - The Phantom Menace,False,6.4,4526.0,2628,120915
5252,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",120000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,en,Star Wars: Episode II - Attack of the Clones,"Ten years after the invasion of Naboo, the gal...",14.072511,/2vcNFtrZXNwIcBgH5e2xXCmVR8t.jpg,...,142.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A Jedi Shall Not Know Anger. Nor Hatred. Nor L...,Star Wars: Episode II - Attack of the Clones,False,6.4,4074.0,5378,121765
10085,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",113000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",http://www.starwars.com/films/star-wars-episod...,en,Star Wars: Episode III - Revenge of the Sith,"Years after the onset of the Clone Wars, the n...",13.165421,/tgr5Pdy7ehZYBqBkN2K7Q02xgOb.jpg,...,140.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The saga is complete.,Star Wars: Episode III - Revenge of the Sith,False,7.1,4200.0,33493,121766
12905,False,NaN,8500000,"[{'id': 53, 'name': 'Thriller'}, {'id': 16, 'n...",http://www.starwars.com/clonewars/site/index.html,en,Star Wars: The Clone Wars,Set between Episode II and III the Clone Wars ...,8.524857,/xd6yhmtS6mEURZLwUDT5raEMbf.jpg,...,98.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Star Wars: The Clone Wars,False,5.8,434.0,61160,1185834
15483,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,en,Empire of Dreams: The Story of the Star Wars T...,Empire of Dreams: The Story of the Star Wars T...,5.796763,/nvYb5G5p94dERKNrALKFNSdHS4O.jpg,...,151.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Empire of Dreams: The Story of the Star Wars T...,False,7.1,22.0,79006,416716
26821,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.starwars.com/films/star-wars-episod...,en,Star Wars: The Force Awakens,Thirty years after defeating the Galactic Empi...,31.626013,/weUSwMdQIa3NaXVzwUoIIcAi85d.jpg,...,136.0

In [88]:
peliculas_star_wars=movies_metadata.loc[movies_metadata.original_title.str.contains('Star Wars|The Empire Strikes Back|Return of the Jedi'),:].ratingsId.values

In [89]:
#Filtramos usuarios que calificaron positivamente dichas películas
aux=ratings_metadata.loc[(ratings_metadata.movieId.isin(peliculas_star_wars))&(ratings_metadata.rating>3),['userId','rating']]
aux=aux.groupby('userId').agg({'rating': ['mean', 'count']})
aux
# aux.reset_index(inplace=True,drop=True)
aux.sort_values(by=('rating','count'),ascending=False)

rating      
            mean count
userId                
199     4.571429     7
287     4.357143     7
176     4.166667     6
607     4.166667     6
260     5.000000     6
...          ...   ...
250     3.500000     1
254     4.000000     1
474     4.000000     1
258     4.000000     1
339     4.000000     1

[307 rows x 2 columns]

In [90]:
#Tomamos de referencia al usuario 199
peliculas_usuario_f=ratings_metadata.loc[(ratings_metadata.userId==199)&(ratings_metadata.movieId.isin(peliculas_star_wars)),:]
peliculas_usuario_f

,userId,movieId,rating,timestamp
27018,199,260,5.0,1240236363
27048,199,1196,5.0,1214669103
27051,199,1210,5.0,1240236362
27101,199,2628,3.5,1214914653
27144,199,5378,4.5,1244632021
27211,199,33493,5.0,1214916719
27340,199,79006,4.0,1310647351


In [91]:
ratings_metadata

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
...,...,...,...,...
99999,671,6268,2.5,1065579370
100000,671,6269,4.0,1065149201
100001,671,6365,4.0,1070940363
100002,671,6385,2.5,1070979663


In [92]:
peliculas_vistas_usuario=ratings_metadata.loc[(ratings_metadata.userId==199),'movieId']
peliculas_no_vistas_usuario=ratings_metadata.loc[~ratings_metadata.movieId.isin(peliculas_vistas_usuario),'movieId'].unique()

pred=[]
for pelicula in peliculas_no_vistas_usuario:
    pred.append(svd.predict(uid=199, iid=pelicula))


### v1

In [95]:
top_n = get_top_n(pred, n=10)
top_n

defaultdict(list,
            {199: [(1228, 4.789188862564544),
              (3730, 4.766402344200127),
              (1259, 4.737509332313688),
              (3030, 4.706036002744023),
              (2278, 4.637190134336654),
              (5225, 4.627700961804187),
              (106782, 4.579472014006947),
              (78499, 4.5716218480368385),
              (1227, 4.567525258920585),
              (33166, 4.556171860607287)]})

In [96]:
# Predecimos el TOP-10 de recomendaciones para el usuario 199

for uid, user_ratings in top_n.items():
    print([movies_metadata.loc[movies_metadata.ratingsId==iid,'original_title'].values[0] for (iid, _) in user_ratings])

#Se recomiendan películas conocidas, con buena valoración, pero en su mayoría no demasiado similares a Star Wars

['Raging Bull', 'The Conversation', 'Stand by Me', '用心棒', 'Ronin', 'Y tu mamá también', 'The Wolf of Wall Street', 'Toy Story 3', 'Once Upon a Time in America', 'Crash']


### v2

Probamos filtrar películas con algunos géneros en común con la película target

In [113]:
import ast

# Si el campo es un string con formato de lista de dicts
movies_metadata['genres_parsed'] = movies_metadata['genres'].apply(
    lambda x: {d['name'] for d in ast.literal_eval(x)} if isinstance(x, str) else set()
)

# Géneros de Star Wars
sw_genres = set()
for _, row in movies_metadata.loc[
    movies_metadata.ratingsId.isin(peliculas_star_wars)
].iterrows():
    sw_genres.update(row['genres_parsed'])

print(sw_genres)  
# {'Science Fiction', 'Adventure', 'Action', 'Fantasy', ...}


# Filtro de predicciones
def comparte_generos(iid, generos_ref, min_comun=1):
    pelicula = movies_metadata.loc[movies_metadata.ratingsId == iid]
    if pelicula.empty:
        return False
    generos_pelicula = pelicula.iloc[0]['genres_parsed']  # ya es un set
    return len(generos_pelicula & generos_ref) >= min_comun

pred_filtradas = [p for p in pred if comparte_generos(p.iid, sw_genres, min_comun=4)]
top_n_filtrado = get_top_n(pred_filtradas, n=10)

{'TV Movie', 'Comedy', 'Family', 'Action', 'Documentary', 'Thriller', 'Adventure', 'Fantasy', 'Animation', 'Science Fiction'}


In [114]:
for uid, user_ratings in top_n_filtrado.items():
    print([movies_metadata.loc[movies_metadata.ratingsId==iid,'original_title'].values[0] for (iid, _) in user_ratings])

['Peter Pan', 'The Court Jester', 'A Grand Day Out', 'The Princess Bride', 'Kung Fu Panda 3', 'The Lego Movie', 'Lethal Weapon', "Dr. Horrible's Sing-Along Blog", 'Mad Max', 'E.T. the Extra-Terrestrial']


# BIBLIOGRAFÍA

- https://towardsdatascience.com/how-you-can-build-simple-recommender-systems-with-surprise-b0d32a8e4802
- https://surprise.readthedocs.io/
- https://medium.com/mlearning-ai/basic-content-based-recommendation-system-with-python-code-be920b412067
- https://medium.com/analytics-vidhya/content-based-recommender-systems-in-python-2b330e01eb80
- https://towardsdatascience.com/hands-on-content-based-recommender-system-using-python-1d643bf314e4
- https://milvus.io/ai-quick-reference/what-is-implicit-feedback-in-recommender-systems
  
LightFM
- https://towardsdatascience.com/solving-business-usecases-by-recommender-system-using-lightfm-4ba7b3ac8e62


